In [1]:
# %%



#------------------------------------------------ Begin_Librairie ----------------------------------------



import pandas as pd



from bs4 import BeautifulSoup



from time import sleep



from datetime import datetime



from pandas import ExcelWriter



from selenium import webdriver



from selenium.webdriver.common.by import By



import datetime



import os



import re



import pdfplumber



from selenium.webdriver.chrome.service import Service as ChromeService








In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'GB PRA'

print(f"Running{regulatorName} Web Scraping Tool v.1.0")


now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') 



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)
    
# %%


RunningGB PRA Web Scraping Tool v.1.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [4]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



# Define a function to scroll to the bottom of the page

def scroll_to_bottom(driver):

    # Get scroll height

    last_height = driver.execute_script("return document.body.scrollHeight")



    while True:

        # Scroll down to the bottom

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        # Wait to load the page

        sleep(3)
        # Calculate new scroll height and compare with last scroll height

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:

            break

        last_height = new_height

def click_element_by_xpath(driver, xpath):

    # Find the element and click

    element = driver.find_element(By.XPATH, xpath)

    element.click()
    

def click_on_cookies(web_driver):

    try:

        web_driver.find_element(By.XPATH,f'//*[@id="modal-content-id-1"]/footer/div/button[3]').click()
        print('[Success] : Success to Click Cookie')

    except Exception as err:

        print('[ERROR] : Failed to click "I Accept" button on the cookies banner:', err)
        

def scrollinAndClick(xpath,key_press=False):
    if len(xpath) != 0 :
        for times in range(60):
            try:
                driver.find_element(By.XPATH, xpath).click()
                sleep(1)
                break
            except:
                print(f"[ERROR] : trying {times+1}/10 to key press 'DOWN' (scrolling)")
                sleep(1)
                if key_press:                    
                    driver.find_element(By.TAG_NAME, 'body').send_keys(key_press)
        else:   
            raise Exception(f'[ERROR] : Failed scrollin Or Click on xpath element : {xpath}')

def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)[0]}")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/{wait_time*2} s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )


# Function to filter out text without numbers
def filter_text_with_numbers(text_list):
    return [text for text in text_list if any(char.isdigit() for char in text)]

# Function to remove content after the number
def remove_content_after_number(text_list):
    import re
    return [re.sub(r'(\d+).*', r'\1', text) for text in text_list]



In [5]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

# Find the inner web link into Browser-devTools-Network, because use the link will not show cookie, avoid one click

# mainAddress = 'https://registres-public.lautorite.qc.ca/1A/Regs.RegistreWeb.Web/en/InstitutionsFinancieres'

regdict={

         'GB PRA 1': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
          
        # #  'GB PRA 13': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',

        # #   'GB PRA 2': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
        
        
          'GB PRA 3': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
        
        # # # 'GB PRA 4': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
        
        # # # 'GB PRA 5': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
        
        # # # 'GB PRA 6': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
        
        # # # 'GB PRA 7': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
        
        # # # 'GB PRA 8': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
        
         'GB PRA 9': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
         'GB PRA 12': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
        
        
        # #'GB PRA 10': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
        
         'GB PRA 11': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
        

          'GB PRA 14': 'https://www.bankofengland.co.uk/prudential-regulation/authorisations/which-firms-does-the-pra-regulate',
        
          'GB PRA 15': 'https://register.fca.org.uk/s/search?predefined=BHC',
        
        }



Typology={
        'GB PRA 1': 'Authorised Insurers Incorporated In Gibraltar',

        'GB PRA 2': 'Authorised UK Insurers',
        
        'GB PRA 13': 'Insurers Incorporated In The EEA With Deemed Part Iva Permission In The SRO',
        
        'GB PRA 3': 'Banks incorporated in Gibraltar entitled to accept deposits through a branch in the UK',
        
        'GB PRA 4': 'Banks incorporated in the EEA entitled to accept deposits in the UK while in the Temporary Permissions Regime (TPR)',
        
        'GB PRA 5': 'Banks incorporated in the EEA entitled to accept deposits through a branch in the UK while in Supervised Run Off (SRO)',
        
        'GB PRA 6': 'Banks incorporated in the United Kingdom',
        
        'GB PRA 7': 'Banks incorporated outside the EEA authorised to accept deposits through a branch in the UK',
        
        'GB PRA 8': 'Banks incorporated outside the UK authorised to accept deposits through a branch in the UK',
        
        'GB PRA 9': 'Building Societies incorporated in the UK',
        
        'GB PRA 10': 'Building Societies incorporated in the United Kingdom (based on PRA authorisation status)',
        
        'GB PRA 11': 'Designated Investment Firms',
        
        'GB PRA 12': 'Individual RFBs',
        
        'GB PRA 14': 'UK Authorised Credit Unions',
        
        'GB PRA 15': 'Parent Financial Holding Companies/Parent Mixed Financial Holding Companies',
        }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}



processdate = now.strftime('%Y-%m-%d')

In [6]:

# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k,reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")   
    # List of Parent Financial Holding Companies/Parent Mixed Financial Holding Companies
    # Except List code 15, the rest of list need to be cope with PDF document
    if reg == 'GB PRA 15':
        driver.get(regdict[reg])
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        click_on_cookies(driver)
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        click_element_by_xpath(driver, '//*[@id="bhc-data-resultcountselect-button"]')
        print(f"[INFO] : Click Show All Button _({reg})_ ")  
        driver.maximize_window()
        sleep(4)
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find("table")
        rows = table.find_all('tr', class_='data-table__row')
        
        filtered_rows = []
        for row in rows:
            tds = row.find_all('td')
            # Check if any td element has non-empty text
            if any(td.text.strip() != '' for td in tds):
                filtered_rows.append(row)
        print(f'[INFO] : - Data Scrapping = {len(filtered_rows)} | {reg}')    
                
        for i,row in enumerate(filtered_rows):
            tds = row.find_all('td')
            
            for i, td in enumerate(tds):
                if td.text!='':
                    if i == 0:
                        sqldict['Name'].append(tds[i].find_all('span')[-1].text.strip())
                    elif i ==1:
                        sqldict['InternalID_1'].append(tds[i].find_all('span')[-1].text.strip())
                        sqldict['InternalID_1_type'].append('Reference Number')
                    elif i ==2:
                        sqldict['RegulationType'].append('Regulated')
                    elif i ==3:
                        sqldict['RegulationDate'].append(tds[i].find_all('span')[-1].text.strip())
                    elif i ==4:
                        sqldict['CancellationDate'].append(tds[i].find_all('span')[-1].text.strip())
                    elif i ==5:
                        cntry = tds[i].find_all('span')[-1].text.strip()
                        if cntry.upper() == 'UNITED KINGDOM' :
                             sqldict['Cntry'].append('GB')
                        else:
                            sqldict['Cntry'].append(cntry)

            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(Typology[reg])
            sqldict = bourange_same_length_array(sqldict)
    elif reg == 'GB PRA 1' or reg == 'GB PRA 2' or  reg == 'GB PRA 13':
        driver.get(regdict[reg])
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        try:
            # try to click cookie
            click_element_by_xpath(driver,'/html/body/div/div[1]/div/div/table/tbody/tr[2]/td[3]/button')
            print(f"[INFO] : Click Cookie _({reg})_ ")  
        except:
            pass
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Click Insurance Sector
        #click_element_by_xpath(driver, '//*[@id="main-content"]/section[3]/div/div[1]/div[2]/ul/li[5]/a')
        sleep(4)
        print(f"[INFO] : {Typology[reg]} _({reg})_ ")  
        #click_element_by_xpath(driver,'//*[@id="main-content"]/section[8]/div/div[1]/div[2]/ul')
        #button = driver.find_element(By.XPATH, "//a[contains(text(), 'List of UK insurers')]")
        button = driver.find_element(By.XPATH, "//a[contains(text(), 'List of PRA-regulated insurers')]")
        sleep(5)
        button.click()
        sleep(5)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        # for times in range(50):
        #     dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        #     if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload') or not dl_files[0].endswith('.pdf'):
        #         break
        #     else:
        #         raise Exception(f'{regulatorName} - Failed to download file - '+reg)
        
        sleep(3)
        
        df = pd.read_csv(dl_files[0])
        df = df.fillna('')
        # Use str.contains to filter rows where "Firm Name" contains 'Insurance'
        indices = df[df["BANK OF ENGLAND"].str.contains("insurer", case=False, na=False)].index

        for i in range(1, len(indices)):
            start = indices[i]
            end = indices[i+1] if i < len(indices)-1 else None  # Last segment goes to the end
            segment = df.iloc[start:end]
            list_name = segment.iloc[0,0]
            print(f'Extract data from ' + segment.iloc[0,0])
            segment = segment[~((segment['Unnamed: 1'].str.strip() == '') & (segment['Unnamed: 2'].str.strip() == ''))]
            # Reset the index of the dataframe
            segment.reset_index(drop=True, inplace=True)
            # Assign the first row as the column headers
            segment.columns = segment.iloc[0]
            # Drop the first row as it is now the header
            segment = segment[1:]
            # Drop the third column
            segment = segment.iloc[:, :2]
            # Remove rows where "Firm Name" is empty
            segment =  segment[segment["Firm Name"].str.strip() != '']
            for name,frn in zip(segment['Firm Name'],segment['FRN']):
                sqldict['Name'].append(str(name))
                sqldict['InternalID_1'].append(frn)
                sqldict['InternalID_1_type'].append('FRN')
                sqldict['ListProcessDate'].append(processdate)
                if 'Gibraltar' in list_name:
                    reg = 'GB PRA 1'
                    
                elif 'SRO' in list_name:
                    reg = 'GB PRA 13'
                else:
                    reg = 'GB PRA 2'
                sqldict['ListName'].append(Typology[reg])  
                sqldict['RegCtry'].append(reg.split(' ')[0]) 
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])          
                sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
            
        ### Old code for pdf file
        # click_element_by_xpath(driver,'//*[@id="item-10"]/div[1]/h3/button')
        # sleep(3)
        # file = driver.find_element(By.XPATH, '//*[@id="main-content"]/section[8]/div/div[1]/div[2]/ul/li/a').click()
        # sleep(5)
        # for times in range(50):
        #     dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        #     if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload') or not dl_files[0].endswith('.pdf'):
        #         break
        #     else:
        #         raise Exception(f'{regulatorName} - Failed to download file - '+reg)
        # pages_text = list()
        # bold_lines = list()
        # with pdfplumber.open(dl_files[0]) as pdf:
        #     for page in pdf.pages:
        #         bold_text = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" in obj["fontname"]).extract_text()
        #         if bold_text is not None:
        #             bold_lines.extend([ele.strip() for ele in bold_text.split('\n') if len(ele) > 0])            
        #         pages_text.append(page.extract_text().strip())

        # all_text = '\n'.join(pages_text)
        # publish_date  = ''
        # for i in os.listdir(tempfolder)[0].split('-')[-2:]:
        #     publish_date+=i
        # lines = [ele.strip() for ele in all_text.split('\n') if len(ele.strip()) > 0]
        
        
                    
        # listNames = []
        # page_title = bold_lines[0]
        # for line in bold_lines:
        #     if line!=page_title:
        #         listNames.append(line)     
        
        # filtered_list = filter_text_with_numbers(lines)
        # infos = []

        # for info in filtered_list:
        #     if ('LIST OF INSURERS AS COMPILED BY THE BANK OF ENGLAND' not in info) and ('Page' not in info):
        #         infos.append(info)
        # clean_info = remove_content_after_number(infos)
        # sectors = []
        # for index,data in enumerate(clean_info):
        #     if index<len(clean_info)-1:
        #         if clean_info[index+1][0].upper()<clean_info[index][0].upper():
        #             sectors.append(index) 
                    
        # for index,data in  enumerate(clean_info):
        #     names = ''
        #     id_FRN = data.split(' ')[-1]
        #     for name in data.split(' ')[0:-1]:
        #         names+=name+' '
        #     sqldict['InternalID_1'].append(id_FRN)
        #     sqldict['InternalID_1_type'].append('FRN')
        #     sqldict['Name'].append(names)
        #     sqldict['ListProcessDate'].append(processdate)
        #     sqldict['RegCtry'].append(reg.split(' ')[0]) 
        #     sqldict['RegCode'].append(reg.split(' ')[1])

        #     if index<=sectors[0]:
        #         sqldict['ListCode'].append('2')
        #         sqldict['ListName'].append(listNames[0])
        #     elif index <=sectors[1]:
        #         sqldict['ListCode'].append('1')
        #         sqldict['ListName'].append(listNames[1])
        #     elif index >sectors[1]:
        #         sqldict['ListCode'].append('13')
        #         if 'Supervised Run' in listNames[2]:
        #             listNames[2]='''Insurers incorporated in the EEA entitled to carry out contracts of insurance through a branch in the UK while in Supervised Run Off (SRO) '''
        #         sqldict['ListName'].append(listNames[2])

        #     sqldict['RegulationType'].append('Regulated')
        #     sqldict['RegulationDate'].append(publish_date[:-4]) 
        
        
            
        for rem in os.listdir(tempfolder):
            os.remove(os.path.join(tempfolder, rem)) 
    elif reg == 'GB PRA 9' or reg == 'GB PRA 12':
        
        driver.get(regdict[reg])
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        try:
            # try to click cookie
            click_element_by_xpath(driver,'/html/body/div/div[1]/div/div/table/tbody/tr[2]/td[3]/button')
            print(f"[INFO] : Click Cookie _({reg})_ ")  
        except:
            pass
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Click Building Societies Sector
        #click_element_by_xpath(driver, '//*[@id="main-content"]/section[3]/div/div[1]/div[2]/ul/li[2]/a')
        print(f"[INFO] : Click {Typology[reg]} _({reg})_ ")  
        sleep(3)
        
        
        if reg == 'GB PRA 9':
            # file = driver.find_element(By.XPATH, '//*[@id="main-content"]/section[5]/div/div[1]/div[2]/ul/li/a').click()
            button = driver.find_element(By.XPATH, "//a[contains(text(), 'List of PRA-regulated Building Societies')]")
            sleep(5)
            button.click()
            sleep(5)
            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
                
        elif reg  == 'GB PRA 12':
            # file = driver.find_element(By.XPATH, '//*[@id="main-content"]/section[5]/div/div[1]/div[4]/ul/li/a').click()
            button = driver.find_element(By.XPATH, "//a[contains(text(), 'List of ring-fenced bodies')]")
            sleep(5)
            button.click()
            sleep(5)
            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
                           
        
        # for times in range(50):
        #     dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        #     if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
        #         break
        #     sleep(3)
        # else:
        #     raise Exception(f'{regulatorName} - Failed to download file - '+reg)
        
        if reg == 'GB PRA 9':
            df = pd.read_csv(dl_files[0])
            df = df.fillna('')   
            # Use str.contains to filter rows where "Firm Name" contains 'incorporated'df['BANK OF ENGLAND\t(PRA)']
            indices = df[df["BANK OF ENGLAND\t(PRA)"].str.contains("Societies", case=False, na=False)].index
            for i in range(len(indices)):
                start = indices[i]
                end = indices[i+1] if i < len(indices)-1 else None  # Last segment goes to the end
                segment = df.iloc[start:end]
                list_name = segment.iloc[0,0]
                print(f'Extract data from ' + segment.iloc[0,0])
                
                segment = segment[~(segment['Unnamed: 1'].str.strip() == '')]
                # Reset the index of the dataframe
                segment.reset_index(drop=True, inplace=True)
                # Assign the first row as the column headers
                segment.columns = segment.iloc[0]
                # Drop the first row as it is now the header
                segment = segment[1:]
                # Drop the third column
                segment = segment.iloc[:, :2]
                # Remove rows where "Firm Name" is empty
                segment =  segment[segment["Firm Name"].str.strip() != '']
                
                for name,frn in zip(segment['Firm Name'],segment['FRN']):
                    sqldict['Name'].append(str(name))
                    sqldict['InternalID_1'].append(frn)
                    sqldict['InternalID_1_type'].append('FRN')
                    sqldict['ListProcessDate'].append(processdate)
                    if 'PRA' in list_name:
                        reg = 'GB PRA 10'

                    else:
                        reg = 'GB PRA 9'
                    sqldict['ListName'].append(Typology[reg])  
                    sqldict['RegCtry'].append(reg.split(' ')[0]) 
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])          
                    sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)
            ## 'GB PRA 9 Old code for pdf' 
            # pages_text = list()
            # bold_lines = list()
            # with pdfplumber.open(dl_files[0]) as pdf:
            #     for page in pdf.pages:
            #         bold_text = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" in obj["fontname"]).extract_text()
            #         if bold_text is not None:
            #             bold_lines.extend([ele.strip() for ele in bold_text.split('\n') if len(ele) > 0])            
            #         pages_text.append(page.extract_text().strip())

            # all_text = '\n'.join(pages_text)
            # lines = [ele.strip() for ele in all_text.split('\n') if len(ele.strip()) > 0]
            
            # contents = lines[len(bold_lines)+1:]
            
            # for item in lines[len(bold_lines)+1:-1]:
            #     id = item.split(' ')[-1]
            #     name = re.sub(id, '', item)
            #     sqldict['Name'].append(name)
            #     sqldict['ListProcessDate'].append(processdate)
            #     sqldict['InternalID_1'].append(id)
            #     sqldict['InternalID_1_type'].append('InternalID')
            #     sqldict['RegCtry'].append(reg.split(' ')[0]) 
            #     sqldict['RegCode'].append(reg.split(' ')[1])
            #     sqldict['ListCode'].append(reg.split(' ')[-1])
            #     sqldict['ListName'].append(Typology[reg])

            #     sqldict['RegulationType'].append('Regulated')
            #     sqldict['RegulationDate'].append(lines[-1].split(' ')[0])
            for rem in os.listdir(tempfolder):
                os.remove(os.path.join(tempfolder, rem)) 
                
        elif reg  == 'GB PRA 12':
            publish_date  = ''
            pages_text = list()
            bold_lines = list()
            table = list()
            for i in os.listdir(tempfolder)[0].split('-')[-2:]:
                publish_date+=i
            with pdfplumber.open(dl_files[0]) as pdf:
                for page in pdf.pages:
                    tables = page.extract_tables(table_settings={})
                    for index,table in enumerate(tables[0]):
                        if table == ['',None,'',None]:
                            pass
                        else:
                            if table[0] == 'Banking Group':
                                pass
                            else:
                                #print(table)
                                if table[0] is not None:
                                    banking_group = table[0]
                                    RFBs = table[-1]
                                else:
                                    RFBs = table[-1]
                                #print(banking_group)
                                sqldict['Name - Mother Company'].append(banking_group)
                                #print(RFBs)
                                sqldict['Name'].append(RFBs)
                                sqldict['RegCtry'].append(reg.split(' ')[0]) 
                                sqldict['RegCode'].append(reg.split(' ')[1])
                                sqldict['ListCode'].append(reg.split(' ')[-1])
                                sqldict['ListName'].append(Typology[reg])
                                sqldict['ListProcessDate'].append(processdate)
                                sqldict['RegulationType'].append('Regulated')
                                sqldict['RegulationDate'].append(publish_date[:-4])    
                            sqldict = bourange_same_length_array(sqldict)

        
            for rem in os.listdir(tempfolder):
                os.remove(os.path.join(tempfolder, rem))         
    elif reg == 'GB PRA 14':
        driver.get(regdict[reg])
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        try:
            # try to click cookie
            click_element_by_xpath(driver,'/html/body/div/div[1]/div/div/table/tbody/tr[2]/td[3]/button')
            print(f"[INFO] : Click Cookie _({reg})_ ")  
        except:
            pass
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Click Insurance Sector
        click_element_by_xpath(driver, '//*[@id="main-content"]/section[3]/div/div[1]/div[2]/ul/li[7]/a')
        print(f"[INFO] : {Typology[reg]} _({reg})_ ") 
        sleep(3)
        button = driver.find_element(By.XPATH, "//a[contains(text(), 'List of authorised credit unions')]")
        sleep(5)
        button.click()
        sleep(5)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            
        sleep(3)
        df = pd.read_csv(dl_files[0])
        df = df.fillna('')
        # Use str.contains to filter rows where "Firm Name" contains 'incorporated'
        indices = df[df["BANK OF ENGLAND"].str.contains("incorporated", case=False, na=False)].index
        for i in range(len(indices)):
            start = indices[i]
            end = indices[i+1] if i < len(indices)-1 else None  # Last segment goes to the end
            segment = df.iloc[start:end]
            list_name = segment.iloc[0,0]
            print(f'Extract data from ' + segment.iloc[0,0])
            
            segment = segment[~(segment['Unnamed: 1'].str.strip() == '')]
            # Reset the index of the dataframe
            segment.reset_index(drop=True, inplace=True)
            # Assign the first row as the column headers
            segment.columns = segment.iloc[0]
            # Drop the first row as it is now the header
            segment = segment[1:]
            # Drop the third column
            segment = segment.iloc[:, :2]
            # Remove rows where "Firm Name" is empty
            segment =  segment[segment["Firm Name"].str.strip() != '']
            for name,frn in zip(segment['Firm Name'],segment['FRN']):
                sqldict['Name'].append(str(name))
                sqldict['InternalID_1'].append(frn)
                sqldict['InternalID_1_type'].append('FRN')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['ListName'].append(Typology[reg])  
                sqldict['RegCtry'].append(reg.split(' ')[0]) 
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])          
                sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
        ## Old code for PDF format
        # pages_text = list()
        # bold_lines = list()
        # with pdfplumber.open(dl_files[0]) as pdf:
        #     for page in pdf.pages:
        #         bold_text = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" in obj["fontname"]).extract_text()
        #         if bold_text is not None:
        #             bold_lines.extend([ele.strip() for ele in bold_text.split('\n') if len(ele) > 0])            
        #         pages_text.append(re.sub(bold_text,'',page.extract_text().strip()))


        # all_text = '\n'.join(pages_text)
        # lines = [ele.strip() for ele in all_text.split('\n') if len(ele.strip()) > 0]
        # date = lines[-1]
        # for item in lines[3:-1]:
        #     #print(item)
        #     id = item.split(' ')[-1]
        #     if len(id)>3:
        #         name = re.sub(id, '', item)
        #         # print(name)
        #         # print(id)
        #         sqldict['Name'].append(name)
        #         sqldict['ListProcessDate'].append(processdate)
        #         sqldict['InternalID_1'].append(id)
        #         sqldict['InternalID_1_type'].append('InternalID')
        #         sqldict['RegCtry'].append(reg.split(' ')[0]) 
        #         sqldict['RegCode'].append(reg.split(' ')[1])
        #         sqldict['ListCode'].append(reg.split(' ')[-1])
        #         sqldict['ListName'].append(Typology[reg])

        #         sqldict['RegulationType'].append('Regulated')
        #         sqldict['RegulationDate'].append(date.split(' ')[0])
        for rem in os.listdir(tempfolder):
                os.remove(os.path.join(tempfolder, rem))        
    elif reg == 'GB PRA 11':
        driver.get(regdict[reg])
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        try:
            # try to click cookie
            click_element_by_xpath(driver,'/html/body/div/div[1]/div/div/table/tbody/tr[2]/td[3]/button')
            print(f"[INFO] : Click Cookie _({reg})_ ")  
        except:
            pass
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Click Investment Sector
        click_element_by_xpath(driver, '//*[@id="main-content"]/section[3]/div/div[1]/div[2]/ul/li[6]/a')
        print(f"[INFO] : {Typology[reg]} _({reg})_ ")  
        sleep(3)
        button = driver.find_element(By.XPATH, "//a[contains(text(), 'List of designated firms')]")
        sleep(5)
        button.click()
        sleep(5)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        pages_text = list()
        bold_lines = list()
        with pdfplumber.open(dl_files[0]) as pdf:
            for page in pdf.pages:
                bold_text = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" in obj["fontname"]).extract_text()
                if bold_text is not None:
                    bold_lines.extend([ele.strip() for ele in bold_text.split('\n') if len(ele) > 0])            
                pages_text.append(re.sub(bold_text,'',page.extract_text().strip()))


        all_text = '\n'.join(pages_text)
        lines = [ele.strip() for ele in all_text.split('\n') if len(ele.strip()) > 0]
        date = ''
        for i in bold_lines[0].split(' ')[-3:]:
            date+=i+' '
        
        for line in lines[:-1]:
            id = line.split(' ')[0]
            name = re.sub(id,'',line)
            
            #print(name)
            sqldict['Name'].append(name)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['InternalID_1'].append(id)
            sqldict['InternalID_1_type'].append('FRN')
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['ListName'].append(Typology[reg])

            sqldict['RegulationType'].append('Regulated')
            sqldict['RegulationDate'].append(date)
        for rem in os.listdir(tempfolder):
            os.remove(os.path.join(tempfolder, rem))        
    elif reg == 'GB PRA 3':
        driver.get(regdict[reg])
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        try:
            # try to click cookie
            click_element_by_xpath(driver,'/html/body/div/div[1]/div/div/table/tbody/tr[2]/td[3]/button')
            print(f"[INFO] : Click Cookie _({reg})_ ")  
        except:
            pass
        sleep(4)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Click Bank Sector
        #click_element_by_xpath(driver, '//*[@id="main-content"]/section[4]/div/div[1]/div[2]/ul/li/a')
        print(f"[INFO] : {Typology[reg]} _({reg})_ ")  
        sleep(3)
        button = driver.find_element(By.XPATH, "//a[contains(text(), 'List of PRA-regulated Banks')]")
        sleep(5)
        button.click()
        sleep(5)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        
        sleep(3)
        
        df = pd.read_csv(dl_files[0])
        df = df.fillna('')
        # Use str.contains to filter rows where "Firm Name" contains 'incorporated'
        #indices = df[df["BANK OF ENGLAND"].str.contains("incorporated", case=False, na=False)].index
        indices = df[df["BANK OF ENGLAND (PRA)"].str.contains("incorporated", case=False, na=False)].index
        for i in range(len(indices)):
            start = indices[i]
            end = indices[i+1] if i < len(indices)-1 else None  # Last segment goes to the end
            segment = df.iloc[start:end]
            list_name = segment.iloc[0,0]
            print(f'Extract data from ' + segment.iloc[0,0])
            
            segment = segment[~(segment['Unnamed: 1'].str.strip() == '')]
            # Reset the index of the dataframe
            segment.reset_index(drop=True, inplace=True)
            # Assign the first row as the column headers
            segment.columns = segment.iloc[0]
            # Drop the first row as it is now the header
            segment = segment[1:]
            # Drop the third column
            segment = segment.iloc[:, :2]
            # Remove rows where "Firm Name" is empty
            segment =  segment[segment["Firm Name"].str.strip() != '']
            
            for name,frn in zip(segment['Firm Name'],segment['FRN']):
                sqldict['Name'].append(str(name))
                sqldict['InternalID_1'].append(frn)
                sqldict['InternalID_1_type'].append('FRN')
                sqldict['ListProcessDate'].append(processdate)
                if 'Gibraltar' in list_name:
                    reg = 'GB PRA 3'
                elif 'TPR' in list_name:
                    reg = 'GB PRA 4'            
                elif 'SRO' in list_name:
                    reg = 'GB PRA 5'
                elif 'outside' in list_name and 'EEA' not in list_name:
                    reg = 'GB PRA 8'
                elif 'outside' in list_name and 'EEA' in list_name:
                    reg = 'GB PRA 7'
                else:
                    reg = 'GB PRA 6'
                sqldict['ListName'].append(Typology[reg])  
                sqldict['RegCtry'].append(reg.split(' ')[0]) 
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])          
                sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
        ## Old code for .pdf file
        # pages_text = list()
        # bold_lines = list()
        # categorys = []
        # with pdfplumber.open(dl_files[0]) as pdf:
        #     for page in pdf.pages:
        #         bold_text_category = ''
        #         for obj in page.chars:
        
        #             if obj["object_type"] == "char" and "Bold" in obj["fontname"]:
        #                 #print(obj)
        #                 #print(obj["fontname"])  # Print the font name
        #                 if obj["text"] and obj['top']>50:
        #                     bold_text_category+=obj["text"]
        #         categorys.append(bold_text_category)
        #         pages_text.append(page.extract_text().strip())


        # all_text = '\n'.join(pages_text)
        # lines = [ele.strip() for ele in all_text.split('\n') if len(ele.strip()) > 0]
        # publish_date  = os.listdir(tempfolder)[0].split('-')[-1].split('.')[0]

        # clear_categorys = []
        # for i in categorys:
        #     if i!='':
        #         clear_categorys.append(i)
        # print(clear_categorys)
        # print(len(clear_categorys))

        # page_category = []
        # for index, line in enumerate(lines):
        #     if any(clear_category.strip() in line for clear_category in clear_categorys):
        #         page_category.append(index)

                            
        # for i in range(len(page_category)):
        #     print(i)
        #     print(f' ================= {lines[page_category[i]]} ===================')
            
        #     if 'gibraltar' in lines[page_category[i]].lower():
                
        #         reg = 'GB PRA 3'
        
        #         print(f'Range {page_category[i]+1} to {page_category[i+1]}')
        #         #print(lines[page_category[i]+1:page_category[i+1]-3])
        #         contents = lines[page_category[i]+1:page_category[i+1]]
        #         for content in contents[:-3]:
        #             id = content.split(' ')[-1]
        #             name = re.sub(id,'',content)
        #             sqldict['InternalID_1'].append(id)
        #             sqldict['InternalID_1_type'].append('InternalID')
        #             sqldict['Name'].append(name)
        #             sqldict['ListProcessDate'].append(processdate)

        #             sqldict['RegCtry'].append(reg.split(' ')[0]) 

        #             sqldict['RegCode'].append(reg.split(' ')[1])

        #             sqldict['ListCode'].append(reg.split(' ')[-1])
        #             sqldict['ListName'].append(Typology[reg])

        #             sqldict['RegulationType'].append('Regulated')

        #             sqldict['RegulationDate'].append(publish_date)
            
        #     elif 'Supervised Run Off' in lines[page_category[i]]:
                
        #         reg = 'GB PRA 5'
                
        #         print(f'Range {page_category[i]+1} to End')
        #         #print(lines[page_category[i]+1:page_category[i+1]-3])
        #         contents = lines[page_category[i]+1:]
        #         for content in contents[:-3]:
        #             id = content.split(' ')[-1]
        #             name = re.sub(id,'',content)
        #             sqldict['Name'].append(name)
        #             sqldict['ListProcessDate'].append(processdate)
        #             sqldict['InternalID_1'].append(id)
        #             sqldict['InternalID_1_type'].append('InternalID')

        #             sqldict['RegCtry'].append(reg.split(' ')[0]) 

        #             sqldict['RegCode'].append(reg.split(' ')[1])

        #             sqldict['ListCode'].append(reg.split(' ')[-1])
        #             sqldict['ListName'].append(Typology[reg])

        #             sqldict['RegulationType'].append('Regulated')
        #             sqldict['RegulationDate'].append(publish_date)
            
        #     elif 'overseas' in lines[page_category[i]].lower():
        #         reg = 'GB PRA 8'
        #         print(f'Range {page_category[i]+1} to {page_category[i+1]}')
        #         contents = lines[page_category[i]+1:page_category[i+1]]
        #         for content in contents[:-3]:
        #             id = content.split(' ')[-1]
        #             try:
        #                 if len(id)>3 and int(id):
        #                     name = re.sub(id,'',content)
        #                     print(name)
        #                     sqldict['Name'].append(name)
        #                     sqldict['ListProcessDate'].append(processdate)
        #                     sqldict['InternalID_1'].append(id)
        #                     sqldict['InternalID_1_type'].append('InternalID')
        #                     sqldict['RegCtry'].append(reg.split(' ')[0]) 

        #                     sqldict['RegCode'].append(reg.split(' ')[1])

        #                     sqldict['ListCode'].append(reg.split(' ')[-1])
        #                     sqldict['ListName'].append(Typology[reg])

        #                     sqldict['RegulationType'].append('Regulated')
        #                     sqldict['RegulationDate'].append(publish_date)
        #             except:
        #                 continue
                

        #     elif i == 0:
        #         reg = 'GB PRA 6'
        #         print(f'Range {page_category[i]+1} to {page_category[i+1]}')
        #         contents = lines[page_category[i]+1:page_category[i+1]]
        #         for content in contents[:-3]:
        #             id = content.split(' ')[-1]
        #             try:
        #                 if len(id)>3 and int(id):
        #                     name = re.sub(id,'',content)
        #                     print(name)
        #                     sqldict['Name'].append(name)
        #                     sqldict['InternalID_1'].append(id)
        #                     sqldict['InternalID_1_type'].append('InternalID')
        #                     sqldict['ListProcessDate'].append(processdate)

        #                     sqldict['RegCtry'].append(reg.split(' ')[0]) 

        #                     sqldict['RegCode'].append(reg.split(' ')[1])

        #                     sqldict['ListCode'].append(reg.split(' ')[-1])
        #                     sqldict['ListName'].append(Typology[reg])

        #                     sqldict['RegulationType'].append('Regulated')
        #                     sqldict['RegulationDate'].append(publish_date)
        #             except:
        #                 continue
  
        
        for rem in os.listdir(tempfolder):
            os.remove(os.path.join(tempfolder, rem)) 
        
            
    sqldict = bourange_same_length_array(sqldict)

[INFO] : Working 1/7 _(GB PRA 1)_ 
[INFO] : Click Cookie _(GB PRA 1)_ 
[INFO] : Authorised Insurers Incorporated In Gibraltar _(GB PRA 1)_ 
Extract data from Insurers authorised to carry out contracts of insurance in the UK
Extract data from Insurers incorporated in Gibraltar authorised to carry out contracts of insurance through a branch or service in the UK
Extract data from Insurers incorporated in the EEA entitled to carry out contracts of insurance through a branch in the UK while in Supervised Run Off (SRO)
[INFO] : Working 2/7 _(GB PRA 3)_ 
[INFO] : Banks incorporated in Gibraltar entitled to accept deposits through a branch in the UK _(GB PRA 3)_ 
Extract data from Banks incorporated in the UK authorised to accept deposits
Extract data from Banks incorporated outside the UK authorised to accept deposits through a branch in the UK
Extract data from Banks incorporated in Gibraltar authorised to accept deposits through a branch or service in the UK
Extract data from Banks incorpor

IndexError: single positional indexer is out-of-bounds

In [11]:
            df = pd.read_csv(dl_files[0])
            df = df.fillna('')   
            # Use str.contains to filter rows where "Firm Name" contains 'incorporated'df['BANK OF ENGLAND\t(PRA)']
            indices = df[df["BANK OF ENGLAND\t(PRA)"].str.contains("Societies", case=False, na=False)].index

In [19]:
indices = df["BANK OF ENGLAND\t(PRA)"].index

In [ ]:
tds[i].find_all('span')[-1].text.strip()

In [ ]:
tds[i].find_all('span')[-1].text.strip()

NameError: name 'tds' is not defined

In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    



C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_9660\3732186060.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df.to_csv('Credits_2025.csv')